In [2]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [3]:
order_items = pd.read_csv(
    "../exports/cleaned_data/order_items_clean.csv"
)

products = pd.read_csv(
    "../exports/cleaned_data/products_clean.csv"
)

In [4]:
order_items.head()

,order_id,product_id,quantity,unit_price
0,1961864118,642612,3,517.03
1,1549769649,378676,1,881.42
2,9185164487,741341,2,923.84
3,9644738826,561860,1,874.78
4,5427684290,602241,2,976.55


In [5]:
order_items.columns.tolist()

['order_id', 'product_id', 'quantity', 'unit_price']

In [6]:
products.head()

,product_id,product_name,category,brand,price,mrp,margin_percentage,shelf_life_days,min_stock_level,max_stock_level
0,153019,Onions,Fruits & Vegetables,Aurora LLC,947.95,1263.93,25.0,3,13,88
1,11422,Potatoes,Fruits & Vegetables,Ramaswamy-Tata,127.16,169.55,25.0,3,20,65
2,669378,Potatoes,Fruits & Vegetables,Chadha and Sons,212.14,282.85,25.0,3,23,70
3,848226,Tomatoes,Fruits & Vegetables,Barad and Sons,209.59,279.45,25.0,3,10,51
4,890623,Onions,Fruits & Vegetables,"Sangha, Nagar and Varty",354.52,472.69,25.0,3,27,55


In [7]:
products.columns.tolist()

['product_id',
 'product_name',
 'category',
 'brand',
 'price',
 'mrp',
 'margin_percentage',
 'shelf_life_days',
 'min_stock_level',
 'max_stock_level']

In [8]:
basket_data = (
    order_items.merge(
        products[
            [
                "product_id",
                "product_name",
                "category"
            ]
        ],
        on="product_id",
        how="left"
    )
)

basket_data.head()

,order_id,product_id,quantity,unit_price,product_name,category
0,1961864118,642612,3,517.03,Pet Treats,Pet Care
1,1549769649,378676,1,881.42,Orange Juice,Cold Drinks & Juices
2,9185164487,741341,2,923.84,Eggs,Dairy & Breakfast
3,9644738826,561860,1,874.78,Orange Juice,Cold Drinks & Juices
4,5427684290,602241,2,976.55,Nuts,Snacks & Munchies


In [9]:
basket = (
    basket_data
    .groupby(
        [
            "order_id",
            "product_name"
        ]
    )["quantity"]
    .sum()
    .unstack()
    .fillna(0)
)

In [10]:
basket = basket.applymap(
    lambda x: 1 if x > 0 else 0
)

basket.head()

product_name,Baby Food,Baby Wipes,Bananas,Biscuits,Bread,Butter,Carrots,Cat Food,Cereal,Cheese,Chips,Chocolates,Cola,Cookies,Cough Syrup,Curd,Detergent,Diapers,Dish Soap,Dog Food,Eggs,Frozen Biryani,Frozen Pizza,Frozen Vegetables,Ice Cream,Iced Tea,Instant Noodles,Lemonade,Lotion,Mango Drink,Mangoes,Milk,Nuts,Onions,Orange Juice,Pain Reliever,Pet Treats,Popcorn,Potatoes,Pulses,Rice,Salt,Shampoo,Soap,Spinach,Sugar,Toilet Cleaner,Tomatoes,Toothpaste,Vitamins,Wheat Flour
order_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
60465,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2237858,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3101265,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5120698,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5512907,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [12]:
from mlxtend.frequent_patterns import apriori

frequent_items = apriori(
    basket,
    min_support=0.01,
    use_colnames=True
)

frequent_items.head()

c:\Users\Srajan\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
0,0.0230,(Baby Food)
1,0.0352,(Baby Wipes)
2,0.0246,(Biscuits)
3,0.0270,(Bread)
4,0.0146,(Butter)


In [13]:
frequent_items.sort_values(
    "support",
    ascending=False
).head(20)

,support,itemsets
28,0.0466,(Pet Treats)
37,0.0410,(Toilet Cleaner)
11,0.0380,(Cough Syrup)
22,0.0376,(Lotion)
14,0.0368,(Dish Soap)
40,0.0366,(Vitamins)
1,0.0352,(Baby Wipes)
6,0.0304,(Cat Food)
31,0.0274,(Pulses)
3,0.0270,(Bread)


In [14]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_items,
    metric="lift",
    min_threshold=1
)

In [15]:
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [16]:
rules.sort_values(
    "lift",
    ascending=False
).head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [17]:
rules_output = rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].copy()

In [18]:
rules_output["antecedents"] = (
    rules_output["antecedents"]
    .apply(lambda x: ", ".join(list(x)))
)

rules_output["consequents"] = (
    rules_output["consequents"]
    .apply(lambda x: ", ".join(list(x)))
)

In [19]:
top_pairs = (
    rules_output
    .sort_values(
        "lift",
        ascending=False
    )
    .head(20)
)

top_pairs

,antecedents,consequents,support,confidence,lift


In [20]:
print("""
Market Basket Insights

• Products appearing together frequently indicate cross-sell opportunities.

• High lift combinations suggest strong customer purchase associations.

• Bundle promotions can be designed around top-performing product pairs.

• Frequently bought together products should be placed strategically in inventory planning.
""")


Market Basket Insights

• Products appearing together frequently indicate cross-sell opportunities.

• High lift combinations suggest strong customer purchase associations.

• Bundle promotions can be designed around top-performing product pairs.

• Frequently bought together products should be placed strategically in inventory planning.



In [21]:
output_path = "../exports/basket_analysis_output/"

In [22]:
frequent_items.to_csv(
    output_path + "frequent_itemsets.csv",
    index=False
)

rules_output.to_csv(
    output_path + "association_rules.csv",
    index=False
)

top_pairs.to_csv(
    output_path + "top_product_pairs.csv",
    index=False
)

In [23]:
top_pairs

,antecedents,consequents,support,confidence,lift


In [24]:
order_items.groupby(
    "order_id"
)["product_id"].count().describe()

count    5000.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: product_id, dtype: float64